In [1]:
import importlib
import models.xvae, models.dec, models.metrics, misc.dataset, misc.helpers
importlib.reload(models.xvae)
importlib.reload(models.dec)
importlib.reload(models.metrics)
importlib.reload(misc.dataset)
importlib.reload(misc.helpers)

from misc.dataset import get_data
from misc.helpers import normalizeRNA
from models.xvae import xvae
from models.dec import XDEC
from models.metrics import calculate_metrics

data = get_data()
x_num = normalizeRNA(data['rnanp'])
x_bin = data['clin']

ae = xvae(s1_input_size=x_num.shape[1], s2_input_size=x_bin.shape[1], ls=16)
ae.build_model()
ae.train(x_num, x_bin)

xdec = XDEC(ae, n_clusters=2)
y_pred, y_proba, centroids, z = xdec.fit(x_num, x_bin, y=data['y'])

acc, ari, nmi = calculate_metrics(data['y'], y_pred)
print('acc={:.3f} ari={:.3f} nmi={:.3f}'.format(acc, ari, nmi))

import os, pandas as pd, torch
os.makedirs('results/manual_run', exist_ok=True)
ae.save_encoder('results/manual_run/encoder_xvae.pt')

emb = pd.DataFrame({'sample_id': data['sample_id']})
for j in range(z.shape[1]): emb['z{}'.format(j)] = z[:, j]
emb['mgs_level'] = data['label_classes'][data['y']]
emb.to_csv('results/manual_run/xdec_latent_embedding.csv', index=False)

# sanity check: confirm the checkpoint is in the new SHAP-compatible format
ckpt_check = torch.load('results/manual_run/encoder_xvae.pt', map_location='cpu')
assert isinstance(ckpt_check, dict) and 's1_input_size' in ckpt_check, (
    "encoder_xvae.pt is still the OLD format (bare state_dict) - "
    "restart the kernel and rerun this cell before using it in the SHAP notebook."
)
print('Encoder checkpoint OK, ready for SHAP:',
     {k: v for k, v in ckpt_check.items() if k != 'state_dict'})

Epoch 12/250 - loss: 2.0789
Epoch 24/250 - loss: 0.8987
Epoch 36/250 - loss: 0.6020
Epoch 48/250 - loss: 0.4455
Epoch 60/250 - loss: 0.4299
Epoch 72/250 - loss: 0.2977
Epoch 84/250 - loss: 0.3009
Epoch 96/250 - loss: 0.2537
Epoch 108/250 - loss: 0.2976
Epoch 120/250 - loss: 0.2017
Epoch 132/250 - loss: 0.1955
Epoch 144/250 - loss: 0.2212
Epoch 156/250 - loss: 0.1899
Epoch 168/250 - loss: 0.1822
Epoch 180/250 - loss: 0.1707
Epoch 192/250 - loss: 0.1630
Epoch 204/250 - loss: 0.1586
Epoch 216/250 - loss: 0.1581
Epoch 228/250 - loss: 0.1536
Epoch 240/250 - loss: 0.1456
Iter 0: acc = 0.53614, nmi = 0.00832, ari = -0.00118 ; loss=0.00000
Iter 50: acc = 0.53614, nmi = 0.00381, ari = -0.00042 ; loss=0.00000
Iter 100: acc = 0.53012, nmi = 0.00135, ari = -0.00236 ; loss=0.00000
Iter 150: acc = 0.50602, nmi = 0.00012, ari = -0.00552 ; loss=0.00000
Iter 200: acc = 0.52410, nmi = 0.00161, ari = -0.00334 ; loss=0.00000
Iter 250: acc = 0.50000, nmi = 0.00007, ari = -0.00575 ; loss=0.00000
Iter 300: a